### IMPORT

In [1]:
from pathlib import Path

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
import rootutils

import sys
rootutils.setup_root(Path(".").resolve(), indicator=".project-root", pythonpath=True)
sys.path.append(str(Path(".").resolve().parent / "src"))

from bev.models.bev_emb import BEVAdapterCNN
from bev.models.bevqa import BEVQA_CNN
from bev.data.bevqa_dataset import BEVQADataset
from bev.models.head import OutputHead
from bev.models.mca import MCALayer
from bev.models.text_emb import TextAdapter
from bev.training.train import train_epoch, val_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"

### PATH

In [2]:
import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf

try:
    initialize(config_path="../configs", version_base=None)
except:
    pass

cfg = compose(config_name="config", overrides=["paths=mini"])

FEAT_DIR = Path(cfg.paths.bev_features_dir)
DICT_DIR = Path(cfg.paths.dict_dir)
GLOVE = Path(cfg.paths.glove_path)

### DATASET

In [3]:
train_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "train_mini",
    json_path=DICT_DIR / "NuScenes_train_questions_mini.json",
    glove=GLOVE
)

In [4]:
val_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "val_mini",
    json_path=DICT_DIR / "NuScenes_val_questions_mini.json",
    glove=GLOVE
)

In [5]:
feat, quest, ans = train_dataset[0]
print(feat.shape, quest.shape, ans)

torch.Size([128, 200, 200]) torch.Size([35, 300]) tensor(26)


### DATALOADER

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [7]:
feat, quest, ans = next(iter(train_dataloader))
print(feat.shape, quest.shape, ans)

torch.Size([4, 128, 200, 200]) torch.Size([4, 35, 300]) tensor([ 0,  3, 23, 18])


### EMBEDDINGS

In [8]:
bev_ad = BEVAdapterCNN() # BEV MAP -> BEV EMB
text_ad = TextAdapter() # [B,35,512] to match BEV emb [B,400,512] 
feat_output = bev_ad(feat)
text_output = text_ad(quest)
print(f"Feat:{feat_output.shape}") # [B,100,512]
print(f"Text:{text_output.shape}") # [B,35,512]

Feat:torch.Size([4, 100, 512])
Text:torch.Size([4, 35, 512])


### MODEL

In [9]:
model = BEVQA_CNN()
out = model(feat, quest)
print(out.shape) # [4, 30]

torch.Size([4, 30])


### TRAIN

In [10]:
model = BEVQA_CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_epoch(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {tr_loss:.4f} - Acc: {tr_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}")

Epoch 01/10 | Train Loss: 1.6598 - Acc: 0.3907 | Val Loss: 6.2366 - Acc: 0.0718
Epoch 02/10 | Train Loss: 1.2223 - Acc: 0.4832 | Val Loss: 6.6691 - Acc: 0.0571
Epoch 03/10 | Train Loss: 1.1672 - Acc: 0.5029 | Val Loss: 6.8518 - Acc: 0.0663
Epoch 04/10 | Train Loss: 1.1212 - Acc: 0.5114 | Val Loss: 7.0848 - Acc: 0.0681
Epoch 05/10 | Train Loss: 1.0942 - Acc: 0.5205 | Val Loss: 7.0399 - Acc: 0.0764
Epoch 06/10 | Train Loss: 1.0749 - Acc: 0.5304 | Val Loss: 6.9406 - Acc: 0.0543
Epoch 07/10 | Train Loss: 1.0411 - Acc: 0.5460 | Val Loss: 7.3911 - Acc: 0.0525
Epoch 08/10 | Train Loss: 1.0277 - Acc: 0.5493 | Val Loss: 7.4766 - Acc: 0.0681
Epoch 09/10 | Train Loss: 1.0064 - Acc: 0.5652 | Val Loss: 7.5466 - Acc: 0.0727
Epoch 10/10 | Train Loss: 0.9747 - Acc: 0.5783 | Val Loss: 7.5146 - Acc: 0.0552
